In [1]:
import pandas as pd
from datetime import date, datetime, timedelta
from sqlalchemy import create_engine, text

engine = create_engine('mysql+pymysql://root:@localhost:3306/portfolio_development')
conpf = engine.connect()

engine = create_engine("sqlite:///c:\\ruby\\portlt\\db\\development.sqlite3")
conlt = engine.connect()

engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()

format_dict = {'amt':'{:,.2f}','net':'{:,.2f}','refund':'{:,.2f}',
              'grs_amt':'{:,.2f}','net_amt':'{:,.2f}','number':'{:,}',
               'ppu':'{:,.4f}'}
year = 2026
pay_date = date.today()
pay_date

datetime.date(2026, 8, 2)

In [2]:
pay_date = date(2026, 7, 31)
pay_date

datetime.date(2026, 7, 31)

In [3]:
sql = '''
SELECT *
FROM dividends
LIMIT 1'''
print(sql)
df = pd.read_sql(sql, conpf)
df.dtypes


SELECT *
FROM dividends
LIMIT 1


id            int64
stock_id      int64
name         object
year          int64
quarter       int64
number        int64
ppu         float64
amt         float64
net         float64
x_date       object
p_date       object
dtype: object

### Restart and run all

In [5]:
sql = '''
SELECT name, p_date, amt, net, amt-net AS refund, year, quarter AS q
FROM dividends
WHERE YEAR(p_date) = %s AND p_date <= '%s'
ORDER BY p_date DESC, name ASC'''
sql = sql % (year, pay_date)
print(sql)

dtl_by_month = pd.read_sql(sql, conpf)
dtl_by_month.style.format(format_dict).hide(axis="index")


SELECT name, p_date, amt, net, amt-net AS refund, year, quarter AS q
FROM dividends
WHERE YEAR(p_date) = 2026 AND p_date <= '2026-07-31'
ORDER BY p_date DESC, name ASC


name,p_date,amt,net,refund,year,q
PROSPECT,2026-07-15,"1,420.00","1,278.00",142.00,2026,2
WHAIR,2026-06-18,"7,170.00","6,453.00",717.00,2026,1
TFFIF,2026-06-16,"3,010.00","2,709.00",301.00,2026,2
IVL,2026-06-12,"1,260.00","1,134.00",126.00,2026,1
CPNREIT,2026-06-11,"15,400.00","13,860.00","1,540.00",2026,1
GVREIT,2026-06-11,"13,427.40","13,427.40",0.00,2026,1
DIF,2026-06-10,"9,999.00","8,999.10",999.90,2026,1
AIMIRT,2026-06-05,184.50,166.05,18.45,2026,1
WHART,2026-06-05,"3,830.00","3,447.00",383.00,2026,1
AWC,2026-05-28,720.00,648.00,72.00,2025,4


In [6]:
sql = '''
SELECT name, p_date, number, ppu, amt, net
FROM dividends
WHERE YEAR(p_date) = %s AND p_date <= '%s'
ORDER BY p_date, name ASC'''
sql = sql % (year, pay_date)
print(sql)

out_by_month = pd.read_sql(sql, conpf)
out_by_month.style.format(format_dict).hide(axis="index")


SELECT name, p_date, number, ppu, amt, net
FROM dividends
WHERE YEAR(p_date) = 2026 AND p_date <= '2026-07-31'
ORDER BY p_date, name ASC


name,p_date,number,ppu,amt,net
3BBIF,2026-03-04,"120,000",0.2241,"26,894.76","24,205.28"
DIF,2026-03-09,"45,000",0.2222,"9,999.00","8,999.10"
GVREIT,2026-03-12,"69,000",0.1783,"12,302.70","11,072.43"
TFFIF,2026-03-18,"20,000",0.1219,"2,438.00","2,194.20"
WHART,2026-03-20,"20,000",0.1960,"3,920.00","3,677.60"
CPNREIT,2026-03-27,"55,000",0.2794,"15,367.00","13,830.30"
WHAIR,2026-03-27,"50,000",0.1755,"8,775.00","7,897.50"
PTTGC,2026-04-24,"6,000",0.5000,"3,000.00","2,700.00"
PTT,2026-04-28,"7,500",1.4000,"10,500.00","9,472.50"
JMT,2026-05-06,"7,000",0.4300,"3,010.00","2,709.00"


In [7]:
dtl_by_month['p_date'] = pd.to_datetime(dtl_by_month['p_date'])
dtl_by_month.dtypes

name              object
p_date    datetime64[ns]
amt              float64
net              float64
refund           float64
year               int64
q                  int64
dtype: object

### Input dividend to YTD Profit

In [9]:
sql = '''
SELECT YEAR(p_date) AS pay_year, SUM(amt) AS grs_amt, SUM(net) AS net_amt, SUM(amt-net) AS refund
FROM dividends
WHERE p_date <= '%s'
GROUP BY pay_year
ORDER BY pay_year DESC'''
sql = sql % pay_date
print(sql)
ttl_by_year = pd.read_sql(sql, conpf)
ttl_by_year.style.format(format_dict).hide(axis="index")


SELECT YEAR(p_date) AS pay_year, SUM(amt) AS grs_amt, SUM(net) AS net_amt, SUM(amt-net) AS refund
FROM dividends
WHERE p_date <= '2026-07-31'
GROUP BY pay_year
ORDER BY pay_year DESC


pay_year,grs_amt,net_amt,refund
2026,"293,134.34","272,721.14","20,413.20"
2025,"456,589.38","402,527.24","54,062.14"
2024,"392,668.45","360,795.14","31,873.31"
2023,"520,124.25","481,979.57","38,144.68"
2022,"708,367.91","661,087.04","47,280.87"
2021,"752,315.10","700,874.50","51,440.60"
2020,"616,228.94","576,030.84","40,198.10"
2019,"552,424.34","510,266.46","42,157.88"
2018,"351,895.60","314,225.58","37,670.02"
2017,"245,981.10","229,487.99","16,493.11"


### This one is by year, quarter to compare by year

In [11]:
sql = '''
SELECT year, SUM(amt) AS grs_amt, SUM(net) AS net_amt, SUM(amt-net) AS refund
FROM dividends
GROUP BY year
ORDER BY year DESC'''
print(sql)
ttl_by_year = pd.read_sql(sql, conpf)
ttl_by_year.style.format(format_dict).hide(axis="index")


SELECT year, SUM(amt) AS grs_amt, SUM(net) AS net_amt, SUM(amt-net) AS refund
FROM dividends
GROUP BY year
ORDER BY year DESC


year,grs_amt,net_amt,refund
2026,"55,700.90","51,473.55","4,227.35"
2025,"501,474.82","462,764.43","38,710.39"
2024,"386,114.05","337,718.24","48,395.81"
2023,"443,011.65","411,178.45","31,833.20"
2022,"646,064.60","602,271.43","43,793.17"
2021,"758,673.21","710,977.48","47,695.73"
2020,"603,486.53","563,137.46","40,349.07"
2019,"601,462.95","557,915.29","43,547.66"
2018,"605,859.60","544,321.18","61,538.42"
2017,"102,343.50","93,374.15","8,969.35"


In [12]:
current_time = datetime.now()
formatted_time = current_time.strftime("%Y:%m:%d %H:%M:%S")
print(formatted_time)

2026:08:02 21:27:04
